¡Hola, Luis!

Mi nombre es Tonatiuh Cruz. Me complace revisar tu proyecto hoy.

Al identificar cualquier error inicialmente, simplemente los destacaré. Te animo a localizar y abordar los problemas de forma independiente como parte de tu preparación para un rol como data-scientist. En un entorno profesional, tu líder de equipo seguiría un enfoque similar. Si encuentras la tarea desafiante, proporcionaré una pista más específica en la próxima iteración.

Encontrarás mis comentarios a continuación - **por favor no los muevas, modifiques o elimines**.

Puedes encontrar mis comentarios en cajas verdes, amarillas o rojas como esta:

<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class="tocSkip"></a>

Éxito. Todo está hecho correctamente.
</div>

<div class="alert alert-block alert-warning">
<b>Comentario del revisor</b> <a class="tocSkip"></a>

Observaciones. Algunas recomendaciones.
</div>

<div class="alert alert-block alert-danger">
<b>Comentario del revisor</b> <a class="tocSkip"></a>

Necesita corrección. El bloque requiere algunas correcciones. El trabajo no puede ser aceptado con comentarios en rojo.
</div>

Puedes responderme utilizando esto:

<div class="alert alert-block alert-info">

In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

zona_1= pd.read_csv('/datasets/geo_data_0.csv')
zona_2= pd.read_csv('/datasets/geo_data_1.csv')
zona_3= pd.read_csv('/datasets/geo_data_2.csv')
#aqui vemos la forma de nuestos df
print(f'asi se ven nuestros dato de zona 1 \n{zona_1.head()}')
print(f'asi se ven nuestros dato de zona 2 \n{zona_2.head()}')
print(f'asi se ven nuestros dato de zona 3 \n{zona_3.head()}\n')

#contamos los valores Nan
print(f'estos son los valores nulos de nuestra zona 1 \n {zona_1.isna().sum()}')
print(f'estos son los valores nulos de nuestra zona 2 \n {zona_2.isna().sum()}')
print(f'estos son los valores nulos de nuestra zona 3 \n {zona_3.isna().sum()}\n')

#revismaos valores duplicados
print(f'valores duplicados zona 1 {zona_1.duplicated().sum()}')

print(f'valores duplicados zona 2 {zona_2.duplicated().sum()}')


print(f'valores duplicados zona 3 {zona_3.duplicated().sum()}')

#separamos las caracteristicas y lo que queremos predecir
features_z1=zona_1[['f0','f1','f2']]
target_z1= zona_1['product']

features_z2=zona_2[['f0','f1','f2']]
target_z2= zona_2['product']

features_z3=zona_3[['f0','f1','f2']]
target_z3= zona_3['product']


state=np.random.RandomState(12345)

#separamos el conjunto de datos 75:25
def model_predic(features, target, test_size):
    x_train, x_valid, y_train, y_valid = train_test_split(features, target, test_size=test_size, random_state=state)
#creamos instancia del modelo y entrenamos
    model= LinearRegression()
    model.fit(x_train, y_train)

    predictions=model.predict(x_valid)
    mse= mean_squared_error(y_valid, predictions)
    rmse=np.sqrt(mse)
    return predictions, y_valid, rmse, predictions.mean()
#realizamos predicciones  y calculamos rmse y el promedio de produccion en unidades
pred_1, y_valid_z1, rmse_z1, pred_z1_mean = model_predic(features_z1, target_z1, .25)

pred_2,  y_valid_z2, rmse_z2, pred_z2_mean = model_predic(features_z2, target_z2, .25)

pred_3, y_valid_z3, rmse_z3, pred_z3_mean = model_predic(features_z3, target_z3, .25)

#mostramos los resultados de las predicciones  en unidades
print(f'el promedio de produccion de la zona 1 es {pred_z1_mean:.2f} miles de barriles')
print(f'el RMSE de la zona 1 es de {rmse_z1:.2f} miles de barriles')

print(f'el promedio de produccion de la zona 2 es {pred_z2_mean:.2f} miles de barriles')
print(f'el RMSE de la zona 2 es de {rmse_z2:.2f} miles de barriles')

print(f'el promedio de produccion de la zona 3 es {pred_z3_mean:.2f} miles de barriles')
print(f'el RMSE de la zona 3 es de {rmse_z3:.2f} miles de barriles\n')


invested_capital= 100000000
ingreso_por_unidad= 4500
n_pozos = 200 
costo_por_pozo= invested_capital/ n_pozos
volumen_minimo_unidades= costo_por_pozo / ingreso_por_unidad

#mostramos la media de produccion en doalres para cada zona
print(f'la media de produccion de zona 1 en USD es de {pred_z1_mean * 4500:,.2f} ')

print(f'la media de produccion de zona 2 en USD es de {pred_z2_mean * 4500:,.2f} ')

print(f'la media de produccion de zona 3 en USD es de {pred_z3_mean * 4500:,.2f} \n ')

print(f'La zona con mayor beneficio es la zona 3 con una prediccion de  {pred_z3_mean * 4500:,.2f} de dolares, por lo tanto es la que mayores beneficios representa mientras que el RSME no varia mucho a la segunda zoma con mas beneficio, pero si cambia bastante la produccion a la zona con menor RSME. Tambien es la zona con el numero mas cercano a la media requerida de 111.1 miles de unidades, contando con {pred_z3_mean:.2f} miles de unidades')

medias= {'zona_1': pred_z1_mean, 'zona_2': pred_z2_mean, 'zona_3': pred_z3_mean}

for zona, media in medias.items():
    print(f'{zona} : media: {media:.2f}, minimo{volumen_minimo_unidades:.1f}, diferencia {media- volumen_minimo_unidades:.2f}')

#funxion para seleccionar la mejor zona de a cuerdo a predicciones y numeros reales, los mejores 200
def seleccion_pozos (y_valid, predictions, ingreso_unidad, inversion):
    ordered= pd.Series(predictions, index=y_valid.index).sort_values(ascending= False)
    ordered= ordered.head(200)
    revenue_zona = y_valid.loc[ordered.index].sum() * ingreso_unidad
    diferencia= revenue_zona - inversion
    return ordered, revenue_zona, diferencia
#llamamos a la fucnion para obtener el id, ingreso, y la diderencia a la media de cada zona  
pozos_ordenados_z1, ingreso_zona_1, diferencia_z1 = seleccion_pozos(y_valid_z1, pred_1, ingreso_por_unidad, invested_capital)

pozos_ordenados_z2, ingreso_zona_2, diferencia_z2= seleccion_pozos(y_valid_z2, pred_2, ingreso_por_unidad, invested_capital)

pozos_ordenados_z3, ingreso_zona_3, diferencia_z3 = seleccion_pozos(y_valid_z3, pred_3, ingreso_por_unidad, invested_capital)


print(f'ganancias para los 200 pozos zona 1 {diferencia_z1:,.2f}')

print(f'ganancias para los 200 pozos zona 2 {diferencia_z2:,.2f}')

print(f'ganancias para los 200 pozos zona 3 {diferencia_z3:,.2f} \n')
    
    
#aplicamos bootstrap con un total de 1000 repeticiones y 500 datos en cada vuelta
    
def bootstrap(y_valid, predictions, ingreso_unidad, inversion, n_muestras=1000):
    preds = pd.Series(predictions, index=y_valid.index)
    ganancias = []
    for i in range(n_muestras):
        sub_preds = preds.sample(n=500, replace=True, random_state=state)
        sub_real = y_valid.loc[sub_preds.index].reset_index(drop=True)
        _, _, ganancia = seleccion_pozos(sub_real, sub_preds.values, ingreso_unidad, inversion)
        ganancias.append(ganancia)
    return pd.Series(ganancias)
        
        
#analisis de ganasias y riesgo reprentado en porcentaje
def resumen(ganancias):
    promedio = ganancias.mean()
    inferior = ganancias.quantile(0.025)
    superior = ganancias.quantile(0.975)
    riesgo = (ganancias < 0).mean() * 100
    return promedio, inferior, superior, riesgo
#llamamos a la funcion para ver los resultados
g1 = bootstrap(y_valid_z1, pred_1, ingreso_por_unidad, invested_capital)
prom1, inf1, sup1, riesgo1 = resumen(g1)
print(f'Zona 1: promedio {prom1:,.2f}, IC 95% [{inf1:,.2f}, {sup1:,.2f}], riesgo {riesgo1:.1f}%')

g2 = bootstrap(y_valid_z2, pred_2, ingreso_por_unidad, invested_capital)
prom2, inf2, sup2, riesgo2 = resumen(g2)
print(f'Zona 2: promedio {prom2:,.2f}, IC 95% [{inf2:,.2f}, {sup2:,.2f}], riesgo {riesgo2:.1f}%')

g3 = bootstrap(y_valid_z3, pred_3, ingreso_por_unidad, invested_capital)
prom3, inf3, sup3, riesgo3 = resumen(g3)
print(f'Zona 3: promedio {prom3:,.2f}, IC 95% [{inf3:,.2f}, {sup3:,.2f}], riesgo {riesgo3:.1f}%')


print(f'\nDebido a las condiciomes especificadas en cuanto al porcentaje de riesgo que esta dispuesta a tomar la emprese con un riesgo de 2.5% la unica zona que cumple con. esta funcion de acuerdo a nuestro estudio seria la zona con un {riesgo2:.1f}%. \n tambien es la de mayor ingreso promedio con {prom2:,.2f} despues de la inversion de 100M de dolares ademas de ser la unica zona donde el IC queda en numeros positivos [{inf2:,.2f}, {sip2:,.2f}]. \n Asi que aunque incialmente se habia recomendado la zona 3, de acuerdo a nuestro estudio la mejor zona para construir los pozos es la zona 2.')


asi se ven nuestros dato de zona 1 
      id        f0        f1        f2     product
0  txEyH  0.705745 -0.497823  1.221170  105.280062
1  2acmU  1.334711 -0.340164  4.365080   73.037750
2  409Wp  1.022732  0.151990  1.419926   85.265647
3  iJLyR -0.032172  0.139033  2.978566  168.620776
4  Xdl7t  1.988431  0.155413  4.751769  154.036647
asi se ven nuestros dato de zona 2 
      id         f0         f1        f2     product
0  kBEdx -15.001348  -8.276000 -0.005876    3.179103
1  62mP7  14.272088  -3.475083  0.999183   26.953261
2  vyE1P   6.263187  -5.948386  5.001160  134.766305
3  KcrkZ -13.081196 -11.506057  4.999415  137.945408
4  AHL4O  12.702195  -8.147433  5.004363  134.766305
asi se ven nuestros dato de zona 3 
      id        f0        f1        f2     product
0  fwXo0 -1.146987  0.963328 -0.828965   27.758673
1  WJtFt  0.262778  0.269839 -2.530187   56.069697
2  ovLUW  0.194587  0.289035 -5.586433   62.871910
3  q6cA6  2.236060 -0.553760  0.930038  114.572842
4  WPMUX -0.5

NameError: name 'sip2' is not defined

<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class="tocSkip"></a>

Excelente trabajo con la carga de la información y su exploración inicial.  

Esto lo complementas con un análisis de registros nulos y valores duplicados  que pudieran sesgar nuestros resultados.

Luis,  implementaste el modelo de regresión lineal de forma excelente para los 3 conjuntos de datos! En este punto ya puedes obervar cuál es el modelo con el mayor $R^2$ y menor RMSE.

Muy buen trabajo con el calculo de las ganacias para cada una de las regiones. Posteriormente este resultados lo complementas con el análisis de riesgos usando bootstrap para probar con diferentes samples y terminar de verificar nuestros resultado